# Pilot: mortality and limited-time images → predicted brain → Llama steering

1. **Show images to TRIBE v2.**
   - **Mortality salience (MS)** and **limited time (LT)** images are shown next to matched controls. Each control keeps the scene and changes the one detail that carries the meaning: a coffin vs a storage chest, an hourglass running out vs one just turned.
   - Compare the predicted brain responses: how close are the MS and LT contrasts?
2. **Turn each brain contrast into a Llama direction.**
   - A cortical map (20,484 vertices) cannot be added to Llama, and Llama never sees images.
   - TRIBE connects the two: its text stream *is* Llama-3.2-3B.
   - So we search for the direction in Llama's residual stream whose effect on TRIBE's predicted cortex best matches the image contrast (`invert_brain_map`).
3. **Steer Llama with those directions.**
   - Ask the in-group-norm and out-group-exclusion items, as in the steering study.
   - Ask open questions about humanity, to see what the direction does in plain language.

**Read before interpreting.**
- This is a pilot: few, generated images. Treat every result as a hypothesis.
- TRIBE predicts an average viewer's passive response, not a lasting mental state.
- The Llama directions are defined by their *predicted brain effect*. Whether they mean "mortality" to Llama is what step 3 tests, against null directions of the same norm.

**Requirements.**
- A GPU with ≥ 16 GB (an RTX A4000 works).
- Install: `pip install -e ".[tribe,plot]" diffusers accelerate`.
- Hugging Face access to `meta-llama/Llama-3.2-3B` and `meta-llama/Llama-3.2-3B-Instruct`.
- About 30 GB of downloads (TRIBE, V-JEPA2, two Llamas, SDXL-Turbo) and about 1–1.5 h on an RTX A4000 (`RUN_CLOSED_LOOP = False` saves 15–20 min).

**Rehearse first:** set `PILOT_DRY_RUN=1`, or set `DRY_RUN = True` below. The whole notebook then runs on CPU in about a minute, with a fake TRIBE and a tiny random Llama. Its numbers are meaningless; the run only checks your setup.

In [ ]:
# ===== 0. Configuration =======================================================
import gc
import logging
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import Image as ShowImage, display
except ImportError:  # outside Jupyter
    display, ShowImage = print, str
logging.basicConfig(level=logging.INFO, format="[%(asctime)s] %(message)s", datefmt="%H:%M:%S")

DRY_RUN = os.environ.get("PILOT_DRY_RUN", "0") == "1"  # True: CPU rehearsal with fakes

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:  # import mortality_tribe without `pip install -e .`
    sys.path.insert(0, str(ROOT))
OUT = ROOT / "outputs" / ("pilot_images_dry" if DRY_RUN else "pilot_images")
OUT.mkdir(parents=True, exist_ok=True)

# --- Step 1: stimuli -------------------------------------------------------------
IMAGE_SOURCE = "procedural" if DRY_RUN else "diffusers"  # "diffusers" | "folder" | "procedural"
IMAGE_DIR = OUT / "images"  # "folder": IMAGE_DIR/<MS|LT>/<family>_s<k>_<target|control>.png
DIFFUSION_MODEL = "stabilityai/sdxl-turbo"
SEEDS = (0, 1)  # 6 families x 2 seeds = 12 pairs per condition
PRESENTATION = dict(lead=6.0, show=3.0, tail=9.0, fps=16)  # grey | image | grey (s)

# --- TRIBE -------------------------------------------------------------------------
TRIBE_CHECKPOINT = "facebook/tribev2"
TRIBE_CACHE = ROOT / "cache"
# e.g. add {"data.text_feature.model_name": r"C:\models\Llama-3.2-3B"} to use a local Llama
TRIBE_CONFIG_UPDATE = {"data.text_feature.batch_size": 1}
EXCLUDE_EARLY_VISUAL = True  # compare and invert outside V1-V4/MT (see 1c)

# --- Step 2: brain map -> Llama direction -------------------------------------------
INVERSION_STEPS = 300
N_NULL_TARGETS = 3  # sign-flipped contrasts: how well can a meaningless map be matched?
RUN_CLOSED_LOOP = True  # steer TRIBE's own Llama, check the predicted brain (~15-20 min)
CLOSED_LOOP_EPS = 0.10
V_SELF_PATH = None  # optional: .npy of the steering notebook's v_self (3072-d)

# --- Step 3: Llama steering ------------------------------------------------------------
STEER_MODEL = "meta-llama/Llama-3.2-3B-Instruct"  # chat needs Instruct; directions come from base
BASE_MODEL = "meta-llama/Llama-3.2-3B"  # for the base-vs-Instruct alignment check
LOCAL_FILES_ONLY = False
ITEMS_PATH = None  # None: built-in English sample (12 + 12); or your items file (JSON/JSONL/YAML)
EPS_LIST = [-0.10, -0.05, 0.05, 0.10]  # relative to the residual-stream norm, as in the steering notebook
EPS_CHAT = 0.10
MAX_NEW_TOKENS = 120
SYSTEM_PROMPT = None
QUESTIONS = [
    "What gives a human life meaning?",
    "Should a community protect its traditions even when some members want to change them?",
    "How should a country treat immigrants who do not share its customs?",
    "If you knew you had only one year left to live, how would you spend it?",
    "What do we owe to strangers?",
    "Is it more important to be remembered after you die or to be kind while you live?",
]


def corr(a, b):
    a, b = a - a.mean(), b - b.mean()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))


print("dry run:", DRY_RUN, "| outputs ->", OUT)

## 1a. Stimuli

The default generates **12 MS pairs and 12 LT pairs** with SDXL-Turbo. Each pair is 6 families × 2 seeds, and both members of a pair use the same seed, so the layout stays similar. Families are listed in `pilot_images.PAIR_PROMPTS`.

- **Check the contact sheet.** If the generator ignored the critical detail (the coffin came out as a chest, the hourglass is full on both sides), delete that pair, set `IMAGE_SOURCE = "folder"` and re-run.
- **Check the low-level table.** Brightness, contrast, edges or colour that differ systematically within pairs would let the *visual* system tell targets from controls for reasons other than meaning.

In [ ]:
from mortality_tribe import pilot_images as pi

if IMAGE_SOURCE == "diffusers":
    manifest = pi.generate_pairs_diffusers(IMAGE_DIR, model_id=DIFFUSION_MODEL, seeds=SEEDS)
elif IMAGE_SOURCE == "folder":
    manifest = pi.scan_image_folder(IMAGE_DIR)
else:
    manifest = pi.draw_pairs_procedural(IMAGE_DIR, seeds=SEEDS)
manifest.to_csv(OUT / "manifest.csv", index=False)
print(manifest.groupby(["cond", "role"]).size().unstack())

display(ShowImage(str(pi.contact_sheet(manifest, OUT / "contact_sheet.png"))))

lowlevel = pi.paired_lowlevel_tests(pi.lowlevel_stats(manifest))
display(lowlevel.round(3))
flagged = lowlevel[lowlevel.p < 0.05]
print("Low-level differences within pairs (p < .05):",
      "none" if flagged.empty else ", ".join(f"{r.cond}:{r.property}" for r in flagged.itertuples()))

## 1b. TRIBE's predicted brain response to each image

Each image is its own timeline: grey 6 s → image 3 s → grey 9 s. An all-grey timeline is the baseline subtracted from every image.

TRIBE's `predict()` row *t* is the predicted BOLD at *t* + 5 s, so the response sits near image onset. The peak is **measured** here, pooled over all images so the window cannot favour any contrast.

The suggested `pred_offset` is what the main pipeline (`configs/default.yaml`, `analysis.pred_offset`) should use.

In [ ]:
import torch

if DRY_RUN:
    from mortality_tribe import pilot_fakes as pf, pilot_steer as ps, stimuli as S

    _stim = S.load_stimuli(ROOT / "stimuli")
    _corpus = pf.corpus_for([i["text"] for i in _stim.primes["NEU"] + _stim.fillers], QUESTIONS,
                            [f"{it.scenario} {it.defensive} {it.inclusive}" for it in ps.load_items(ITEMS_PATH)])
    tiny_llm, tiny_tok = pf.tiny_llama(OUT / "tiny_llama", _corpus)
    tribe = pf.FakeTribe(tiny_llm, tiny_tok, OUT / "tiny_llama")
else:
    from tribev2 import TribeModel

    tribe = TribeModel.from_pretrained(TRIBE_CHECKPOINT, cache_folder=TRIBE_CACHE, config_update=TRIBE_CONFIG_UPDATE)

pres = pi.Presentation(**PRESENTATION)
videos = {"grey": pi.make_image_video(None, OUT / "videos" / "grey.mp4", pres)}
image_keys = [f"{r.pair_id}_{r.role}" for r in manifest.itertuples()]
for key, path in zip(image_keys, manifest.path):
    videos[key] = pi.make_image_video(path, OUT / "videos" / f"{key}.mp4", pres)

responses = pi.predict_timelines(tribe, videos)
peak_lag, lag_curve = pi.estimate_peak_lag(responses, "grey", pres.lead, image_keys)
maps = pi.image_maps(responses, "grey", pres.lead, peak_lag, image_keys)  # rows follow `manifest`
np.save(OUT / "image_maps.npy", maps)

print(f"Image-evoked response peaks {peak_lag:+.0f} s after image onset (predict() time).")
print(f"Suggested analysis.pred_offset for the main pipeline: "
      f"{pi.suggested_pred_offset(peak_lag, pres.show):.0f} (default 5)")
ax = lag_curve.plot(x="rel_onset", y="rms_response", legend=False, figsize=(5, 2.5))
ax.axvspan(0, pres.show, color="0.9")
ax.set_xlabel("s from image onset")
ax.set_ylabel("RMS(image - grey)")
plt.show()

## 1c. How close are mortality and limited time?

**Evoked maps** (target image − grey) share "seeing a picture", so they correlate highly whatever the content. That number is shown but not informative.

The informative comparison is between the **meaning contrasts**, target − control within each pair:

1. **Is each contrast consistent across pairs?** A sign-flip test on the size of the mean map. If MS − control is not consistent, nothing downstream means anything.
2. **How similar are MS − control and LT − control?**
   - `r` is reported with a sign-flip null.
   - Each contrast has a split-half reliability; `r` cannot exceed √(rel_MS · rel_LT).
   - The row **outside early visual** cortex is the one to read. Early visual areas mostly carry pixel-level differences the generator could not hold constant.
3. ROI means (your hypothesis ROIs), the top HCP areas, and cortical maps.

In [ ]:
from mortality_tribe import rois as R

V = maps.shape[1]
mask = pi.cortex_mask_without(pi.EARLY_VISUAL_HCP if (EXCLUDE_EARLY_VISUAL and not DRY_RUN) else [], V)
diffs = pi.pair_differences(maps, manifest)
d_ms, d_lt = diffs["MS"][1], diffs["LT"][1]
c_ms, c_lt = d_ms.mean(0), d_lt.mean(0)
np.savez(OUT / "contrasts.npz", c_ms=c_ms, c_lt=c_lt, mask=mask)

evoked = {c: maps[((manifest.cond == c) & (manifest.role == "target")).to_numpy()].mean(0) for c in ("MS", "LT")}
print(f"Evoked maps (target - grey), whole cortex: r = {corr(evoked['MS'], evoked['LT']):.2f}"
      "  <- mostly 'seeing a picture'")

for name, d in (("MS", d_ms), ("LT", d_lt)):
    rel = pi.contrast_reliability_null(d, mask)
    print(f"{name} - control consistent across {len(d)} pairs? |mean map| = {rel['norm']:.3f}, "
          f"sign-flip p = {rel['p_perm']:.4f}")

similarity = pd.DataFrame([pi.compare_contrasts(d_ms, d_lt), pi.compare_contrasts(d_ms, d_lt, mask)],
                          index=["whole cortex", "outside early visual"])
display(similarity.round(3))
print(f"Top-5% vertices overlap (Dice, outside early visual): "
      f"{pi.top_fraction_overlap(c_ms[mask], c_lt[mask]):.2f}")

roi_idx = R.fake_roi_indices(V) if DRY_RUN else R.hcp_roi_indices()
roi_table = pd.DataFrame({"MS - control": [c_ms[i].mean() for i in roi_idx.values()],
                          "LT - control": [c_lt[i].mean() for i in roi_idx.values()]}, index=list(roi_idx))
display(roi_table.round(4))

if not DRY_RUN:
    from tribev2.plotting import PlotBrainNilearn
    from tribev2.utils import get_topk_rois

    print("MS - control, top HCP areas:", list(get_topk_rois(c_ms, k=10)))
    print("LT - control, top HCP areas:", list(get_topk_rois(c_lt, k=10)))
    plotter = PlotBrainNilearn(mesh="fsaverage5")
    for name, m in (("MS - control", c_ms), ("LT - control", c_lt)):
        vmax = float(np.percentile(np.abs(m), 99))
        plotter.plot_surf(m, views=["left", "right", "medial_left", "medial_right"], cmap="cold_hot",
                          vmin=-vmax, vmax=vmax, symmetric_cbar=True, colorbar=True)
        plt.gcf().suptitle(name)
        plt.show()

fig, ax = plt.subplots(figsize=(3.5, 3.5))
ax.hexbin(c_ms[mask], c_lt[mask], gridsize=40, mincnt=1, cmap="Greys")
ax.set_xlabel("MS - control")
ax.set_ylabel("LT - control")
ax.set_title(f"r = {similarity.loc['outside early visual', 'r']:.2f}")
plt.show()

## 2. From brain map to Llama direction

TRIBE turns Llama-3.2-3B hidden states into predicted cortex. On ordinary, neutral sentences (the NEU primes and fillers of the main experiment), we search for the direction **u** in Llama's residual stream that, when added to those features, changes the predicted cortex most like `MS − control` (and separately `LT − control`).

- **Null directions:** the same search on sign-flipped contrasts, which are maps with the same smoothness but no consistent meaning. `u_MS` is only special if it fits its target clearly better than the nulls fit theirs.
- **Symmetry:** +u and −u should move the brain in opposite directions (the directional vs symmetric split of the steering notebook).
- **Closed loop (2c):** steer TRIBE's *own* Llama with **u** after the decoder layer where TRIBE starts reading. Does the predicted brain move toward the image contrast? This checks the approximation used in the search, which adds **u** directly to TRIBE's features.

In [ ]:
from mortality_tribe import pilot_bridge as pb
from mortality_tribe import stimuli as S

stim = S.load_stimuli(ROOT / "stimuli")
passages = {it["id"]: " ".join(it["text"].split()) for it in stim.primes["NEU"] + stim.fillers}  # neutral text
events = pf.simple_text_events(passages) if DRY_RUN else pb.text_events(passages)
batches = pb.text_batches(tribe, events)
groups = pb.tribe_text_layer_groups(tribe)
INJECT = pb.injection_layer(groups)
D = batches[0].data["text"].shape[2]
print(f"TRIBE reads Llama hidden states {groups} -> steer after decoder layer {INJECT}; hidden size {D}")

In [ ]:
targets = {"MS": c_ms, "LT": c_lt}
inv = {k: pb.invert_brain_map(tribe._model, batches, t, mask=mask, n_steps=INVERSION_STEPS, seed=0)
       for k, t in targets.items()}
null_targets = pb.sign_flip_targets(np.concatenate([d_ms, d_lt]), N_NULL_TARGETS)
nulls = [pb.invert_brain_map(tribe._model, batches, t, mask=mask, n_steps=INVERSION_STEPS, seed=k + 1, verbose=False)
         for k, t in enumerate(null_targets)]
fn = inv["MS"].feature_norm
u_rand = np.random.default_rng(0).normal(size=D)
u_rand /= np.linalg.norm(u_rand)
d_rand = pb.delta_for(tribe._model, batches, u_rand, fn)


def fit_row(name, fit_r, delta):
    return dict(direction=name, fit_r=fit_r, r_with_MS=corr(delta[mask], c_ms[mask]),
                r_with_LT=corr(delta[mask], c_lt[mask]))


fit = pd.DataFrame([fit_row(f"u_{k}", res.r, res.delta) for k, res in inv.items()]
                   + [fit_row(f"u_null{i}", res.r, res.delta) for i, res in enumerate(nulls)]
                   + [fit_row("u_rand", np.nan, d_rand)])
display(fit.round(3))
print("fit_r = how well a direction reproduces its own target; u_MS / u_LT should beat every u_null.")

display(pd.DataFrame({k: pb.symmetry(tribe._model, batches, res.u, fn, targets[k], mask)
                      for k, res in inv.items()}).T.round(3))

VECTORS = {"u_MS": inv["MS"].u, "u_LT": inv["LT"].u, "u_null": nulls[0].u, "u_rand": u_rand}
display(pd.DataFrame([[pb.cosine(VECTORS[a], VECTORS[b]) for b in VECTORS] for a in VECTORS],
                     index=list(VECTORS), columns=list(VECTORS)).round(3))
if V_SELF_PATH:
    v_self = np.load(V_SELF_PATH)
    print(f"cos(u_MS, v_self) = {pb.cosine(VECTORS['u_MS'], v_self):.3f}  (v_self comes from another layer)")
np.savez(OUT / "vectors.npz", inject_layer=INJECT, **VECTORS)

In [ ]:
if RUN_CLOSED_LOOP:
    tribe._model.to("cpu")  # make room for TRIBE's fp32 Llama
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    factory = (lambda cache: pf.FakeTribe(tiny_llm, tiny_tok, OUT / "tiny_llama")) if DRY_RUN else None
    closed = pb.closed_loop_check(TRIBE_CHECKPOINT, TRIBE_CACHE, OUT / "steer_cache", events,
                                  {k: VECTORS[k] for k in ("u_MS", "u_LT", "u_null")}, INJECT, CLOSED_LOOP_EPS,
                                  targets, mask=mask, config_update=TRIBE_CONFIG_UPDATE, factory=factory)
    display(closed.round(3))
    print("Steering TRIBE's own Llama with u_MS should move the predicted brain toward MS - control "
          "(r_MS for u_MS above r_MS for u_null).")

## 3. Steering Llama

The directions are added after decoder layer `INJECT`, the first layer TRIBE reads, at every position. They are scaled relative to the residual-stream norm there, like ε in the steering notebook.

- **Base vs Instruct:** the directions come from the *base* model (TRIBE's), but chat needs *Instruct*. The alignment check shows how similar the two models' residual streams are at that layer on the same text.
- **Defense items:** each item is asked twice with the letters swapped; `delta = logP(defensive) − logP(inclusive)`. `change` is the shift from the unsteered model, paired over items.

In [ ]:
from mortality_tribe import pilot_steer as ps

del batches, tribe
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

if DRY_RUN:
    llm, tok = tiny_llm, tiny_tok
else:
    llm, tok = ps.load_llm(STEER_MODEL, local_files_only=LOCAL_FILES_ONLY)
items = ps.load_items(ITEMS_PATH)
print(len(items), "items:", pd.Series([it.axis for it in items]).value_counts().to_dict())
print("First item (check which option is defensive):", items[0])
h_norm = ps.residual_norm(llm, tok, [ps.mc_text(it, it.defensive, it.inclusive) for it in items[:16]], INJECT)
print(f"Residual-stream norm after layer {INJECT}: {h_norm:.3g} (eps = 0.10 adds a vector of norm {0.1 * h_norm:.3g})")

if not DRY_RUN and STEER_MODEL != BASE_MODEL:
    base_llm, _ = ps.load_llm(BASE_MODEL, local_files_only=LOCAL_FILES_ONLY)
    print(f"Base vs {STEER_MODEL}, same text, after layer {INJECT}:",
          ps.hidden_state_alignment(base_llm, llm, tok, list(passages.values()), INJECT))
    del base_llm
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
sweep = ps.steering_sweep(llm, tok, items, VECTORS, EPS_LIST, INJECT, h_norm)
sweep.to_csv(OUT / "defense_sweep.csv", index=False)
summary = ps.summarize_sweep(sweep)
summary.to_csv(OUT / "defense_summary.csv", index=False)
display(summary.round(3))
display(ps.decompose(summary).round(3))

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
for ax, axis in zip(axes, ["ingroup_norm", "outgroup_exclusion"]):
    for vec, g in summary[summary.axis == axis].groupby("vector"):
        g = pd.concat([g, pd.DataFrame([dict(eps_rel=0.0, change=0.0, se=0.0)])]).sort_values("eps_rel")
        ax.errorbar(g.eps_rel, g.change, yerr=g.se, marker="o", ms=3, capsize=2, label=vec)
    ax.axhline(0, color="k", lw=0.5)
    ax.set_title(axis)
    ax.set_xlabel("eps (relative)")
axes[0].set_ylabel("change in logP(def) - logP(inc)")
axes[1].legend(fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
chat = ps.chat_demo(llm, tok, QUESTIONS, {k: VECTORS[k] for k in ("u_MS", "u_LT", "u_null")}, EPS_CHAT, INJECT,
                    h_norm, system=SYSTEM_PROMPT, max_new_tokens=MAX_NEW_TOKENS)
chat.to_csv(OUT / "chat_demo.csv", index=False)
print(ps.format_chat(chat))

## How to read the results

| Step | It counts only if … |
|---|---|
| 1c | `MS − control` is consistent across pairs (sign-flip p) **outside early visual cortex** |
| 1c | MS–LT similarity: read `r` against its ceiling √(rel_MS · rel_LT), not against 1 |
| 2 | `u_MS` fits its target clearly better than every `u_null`; the closed loop moves the brain toward MS for `u_MS` but not `u_null` |
| 3 | The defense shift is larger for `u_MS` than for `u_null` and `u_rand` at the same ε; it **flips with the sign of ε** (directional ≫ symmetric); answer mass stays high |
| 3 | Chat: a theme (tradition, belonging, time, relationships) appears under +`u_MS` and not under `u_null`; `[!! repetitive]` marks broken text |

**Compare with the steering study.** There, `self_clean` raised *in-group norm* defense while the exclusion axis was not finality-specific. Does `u_MS`, a direction found from *images* through a *brain model*, reproduce that pattern?

**Do not conclude:**
- that people feel or behave this way: TRIBE predicts passive average responses, and Llama is not a person;
- anything from the dry run;
- anything from one seed and 12 pairs. Repeat with new seeds and families before writing it up.